# Universidad Externado de Colombia - Finanzas Cuantitativas
## Laboratorio de Modelos Estocásticos, Eficiencia de Mercado y Gestión de Riesgo (Kibo)

Este notebook contiene el desarrollo matemático, estadístico y computacional de los modelos integrados en la aplicación **Kibo**:
1. **Descarga y Limpieza de Datos**: Canasta diversificada de 10 acciones en USD (mínimo 4 sectores).
2. **4 Modelos Estocásticos de Proyección**:
   - Movimiento Browniano Geométrico (GBM)
   - Modelo de Salto-Difusión de Merton (MJD)
   - Proceso de Reversión a la Media de Ornstein-Uhlenbeck (O-U)
   - Movimiento Browniano Fraccionario (fBm) con Análisis R/S del Exponente de Hurst ($H$)
3. **Evaluación Rolling del GBM (Paso 13 del PDF)**: Ventana móvil de 252 sesiones pronosticando a $H=1$ día, con cálculo de RMSE y MAPE vs pronóstico ingenuo.
4. **Test Econométrico de Comportamiento en Manada (Herding Behavior)**: Metodología de Chang, Cheng & Khorana (CCK 2000) con Desviación Absoluta Transversal ($CSAD_t$) y errores robustos White/HAC.
5. **Medición de Riesgo Integral (Paso 10 del PDF)**: Tres estimadores de VaR al 99% a 10 días (Paramétrico, Histórico y Monte Carlo 25.000 trayectorias) + CVaR (Expected Shortfall).
6. **Pruebas de Estrés Oficiales (Stress Testing)**:
   - Estrés 1: Caída uniforme de mercado (-10%)
   - Estrés 2: Desplome del sector mayoritario (-20%) y contagio (-5%)
   - Estrés 3: Pánico de volatilidad ($	imes 1.5$) y correlación $\rho = 0.80$
7. **Optimización Cuadrática de Mínimo VaR (SLSQP)** con restricciones de asignación individual ($2\% \le w_i \le 20\%$) y sectorial (máximo $35\%$), redondeo a acciones enteras y efectivo residual.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
from scipy import stats, optimize

from src.data_engine import download_market_data, DEFAULT_TICKERS, SECTOR_MAP
from src.stochastic_engine import (
    calibrate_gbm, simulate_gbm,
    calibrate_merton_jump, simulate_merton_jump,
    calibrate_ornstein_uhlenbeck, simulate_ornstein_uhlenbeck,
)
from src.fractional_engine import estimate_hurst_rs, simulate_fractional_brownian_motion
from src.herding_engine import run_herding_analysis
from src.rolling_engine import run_gbm_rolling_evaluation, run_multi_asset_rolling_table
from src.risk_engine import compute_var_comparison_and_stress
from src.portfolio_engine import compute_portfolio_allocation

print('✓ Todos los módulos de Kibo importados con éxito.')

### 1. Descarga y Preparación de Datos (10 Acciones · 6 Sectores)

In [ ]:
prices_df, returns_df, stats_dict = download_market_data(DEFAULT_TICKERS, years=3)
print(f'Dimensiones precios: {prices_df.shape}')
print(f'Rango de fechas: {prices_df.index[0].date()} a {prices_df.index[-1].date()}')
prices_df.tail()

### 2. Calibración y Simulación de los 4 Modelos Estocásticos (Ejemplo con AAPL)

In [ ]:
asset = 'AAPL'
spot_aapl = float(prices_df[asset].iloc[-1])
rets_aapl = returns_df[asset]

# 1. GBM
p_gbm = calibrate_gbm(rets_aapl)
sim_gbm = simulate_gbm(spot_aapl, p_gbm, horizon_days=10, n_paths=3000)

# 2. Merton Jump-Diffusion
p_mjd = calibrate_merton_jump(rets_aapl)
sim_mjd = simulate_merton_jump(spot_aapl, p_mjd, horizon_days=10, n_paths=3000)

# 3. Ornstein-Uhlenbeck
p_ou = calibrate_ornstein_uhlenbeck(prices_df[asset])
sim_ou = simulate_ornstein_uhlenbeck(spot_aapl, p_ou, horizon_days=10, n_paths=3000)

# 4. Browniano Fraccionario (fBm)
hurst_res = estimate_hurst_rs(rets_aapl)
sim_fbm = simulate_fractional_brownian_motion(
    spot_aapl,
    p_gbm['mu_annual'],
    p_gbm['sigma_annual'],
    H=hurst_res['H'],
    horizon_days=10,
    n_paths=3000,
    ticker=asset
)

print(f'GBM: mu={p_gbm["mu_annual"]:.2%}, sigma={p_gbm["sigma_annual"]:.2%}')
print(f'Merton: lambda={p_mjd["lambda"]:.1f} saltos/año, caida salto={p_mjd["mu_j"]:.2%}')
print(f'Ornstein-Uhlenbeck: media={p_ou["mu_ou_price"]:.2f}, vida media={p_ou["half_life_days"]:.0f} dias')
print(f'Browniano Fraccionario: Exponente de Hurst H={hurst_res["H"]:.2f} ({hurst_res["regime"]})')

### 3. Evaluación Rolling del GBM (Paso 13 del PDF)
Ventana móvil de calibración de 252 sesiones pronosticando a $H=1$ día.

In [ ]:
roll_table = run_multi_asset_rolling_table(prices_df, DEFAULT_TICKERS, window_train=252, horizon_H=1)
roll_table

### 4. Test Econométrico de Comportamiento en Manada (Herding CCK 2000)
Estimación de $CSAD_t = \alpha + \gamma_1 |R_{m,t}| + \gamma_2 R_{m,t}^2 + \varepsilon_t$.

In [ ]:
herding_res = run_herding_analysis(returns_df)
print('Veredicto:', herding_res['verdict'])
print(f'gamma_2: {herding_res["gamma2"]:.4f} (t={herding_res["gamma2_t"]:.2f}, p-valor={herding_res["gamma2_p"]:.4f})')
print('Asimetria:', herding_res['asym_insight'])

### 5. Optimización Cuadrática de Mínimo VaR (SLSQP)

In [ ]:
from src.data_engine import get_default_portfolio
portfolio_base = get_default_portfolio()
capital_sim = 100000.0

alloc_res = compute_portfolio_allocation(capital_sim, portfolio_base, returns_df, confidence=99.0, horizon_days=10)
alloc_res['allocation_df']

### 6. Medición de Riesgo: Los 3 Métodos de VaR y Pruebas de Estrés (Paso 10 del PDF)

In [ ]:
df_alloc = alloc_res['allocation_df']
risk_res = compute_var_comparison_and_stress(
    capital=capital_sim,
    weights=alloc_res['w_min_var'],
    returns_df=returns_df,
    sectors=df_alloc['Sector'].tolist(),
    tickers=df_alloc['Ticker'].tolist(),
    confidence=0.99,
    horizon_days=10,
    n_mc_sims=20000
)

print('=== COMPARATIVA DE 3 ESTIMADORES DE VaR 99% (10 DÍAS) ===')
display(risk_res['var_comparison_df'])

print('=== PRUEBAS DE ESTRÉS OFICIALES (STRESS TESTING) ===')
display(risk_res['stress_df'])